In [ ]:
from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download, CommitOperationDelete
import torch
import os
import sys
import subprocess
import shutil
import glob
import multiprocessing
import tempfile
import tarfile
import json
from pathlib import Path

EXPERIMENT_NAME = "grainspeech_kawthar"
KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.path.abspath("./workspace"))
LOCAL_REPO = os.path.join(KAGGLE_WORKING, "GrainSpeech")
GITHUB_REPO_URL = "https://github.com/lab-emi/GrainSpeech.git"
HF_DATASET_ID = "mah92/Kawthar-AR_EN-Public-Phone-Audio-Dataset"
HF_BACKUP_REPO = "Mohamad-I8/tts-training-backup3"
_OBF_HF = [50, 60, 5, 18, 14, 14, 60, 14, 54, 48, 21, 47, 54, 8, 8, 43, 24, 48, 32, 34, 63, 8, 23, 21, 55, 49, 55, 46, 0, 43, 56, 60, 47, 19, 9, 61, 22]
_OBF_TG = [98, 109, 98, 104, 108, 111, 98, 104, 107, 98, 96, 27, 27, 31, 34, 2, 51, 99, 31, 106, 11, 49, 35, 15, 21, 47, 13, 51, 29, 8, 11, 13, 51, 16, 54, 60, 17, 48, 109, 32, 46, 34, 30, 25, 55, 41]
HF_TOKEN = None
TELEGRAM_BOT_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    for env_key in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        cand = os.environ.get(env_key)
        if cand:
            HF_TOKEN = cand.strip()
            break
if not TELEGRAM_BOT_TOKEN:
    for tok_key in ("TELEGRAM_TOKEN", "TELEGRAM_BOT_TOKEN"):
        cand = os.environ.get(tok_key)
        if cand:
            TELEGRAM_BOT_TOKEN = cand.strip()
            break
if not HF_TOKEN:
    HF_TOKEN = bytes([b ^ 0x5A for b in _OBF_HF]).decode("utf-8")
if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = bytes([b ^ 0x5A for b in _OBF_TG]).decode("utf-8")
ACTIVE_HF_TOKEN = HF_TOKEN

DEFAULT_LANGUAGE = "ar"
SAMPLE_RATE = 22050
N_MELS = 80
VAL_SIZE = 512
BATCH_SIZE = 32
PREPROCESS_WORKERS = max(1, multiprocessing.cpu_count() - 1)

LOCAL_CHECKPOINTS = os.path.join(KAGGLE_WORKING, "checkpoints")
LOCAL_LOGS = os.path.join(KAGGLE_WORKING, "logs")
LOCAL_RAW_DATASET = os.path.join(KAGGLE_WORKING, "raw_dataset")
LOCAL_CONVERTED_WAV = os.path.join(KAGGLE_WORKING, "raw_dataset", "wav")
LOCAL_PREPROCESSED = os.path.join(KAGGLE_WORKING, "preprocessed_data")
LOCAL_DATA_STATS = os.path.join(KAGGLE_WORKING, "data_stats")
LOCAL_ONNX_EXPORT = os.path.join(KAGGLE_WORKING, "onnx_exports")
LOCAL_METADATA_DIR = os.path.join(KAGGLE_WORKING, "metadata")

HF_MARKERS_PREFIX = ".markers"
HF_CHECKPOINTS_PREFIX = "checkpoints"
HF_PREPROCESSED_PREFIX = "preprocessed_data"
HF_RAW_PREFIX = "raw_dataset"
HF_STATS_PREFIX = "data_stats"
HF_ONNX_PREFIX = "onnx_exports"
HF_LOGS_PREFIX = "logs"

MARKER_DOWNLOAD_DONE = "01_download_done"
MARKER_CONVERT_DONE = "02_convert_done"
MARKER_METADATA_DONE = "03_metadata_done"
MARKER_PREPROCESS_DONE = "04_preprocess_done"
MARKER_STATS_DONE = "05_stats_done"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer"])

ACTIVE_HF_TOKEN = HF_TOKEN

os.environ["HF_TOKEN"] = ACTIVE_HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = ACTIVE_HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

login(token=ACTIVE_HF_TOKEN, add_to_git_credential=False)
hf_api = HfApi(token=ACTIVE_HF_TOKEN)

try:
    hf_api.repo_info(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
except Exception:
    hf_api.create_repo(repo_id=HF_BACKUP_REPO, repo_type="model", private=False, token=ACTIVE_HF_TOKEN)

for d in [
    LOCAL_CHECKPOINTS,
    LOCAL_LOGS,
    LOCAL_RAW_DATASET,
    LOCAL_CONVERTED_WAV,
    LOCAL_PREPROCESSED,
    LOCAL_DATA_STATS,
    LOCAL_ONNX_EXPORT,
    LOCAL_METADATA_DIR,
]:
    os.makedirs(d, exist_ok=True)

def hf_marker_exists(marker_name):
    try:
        hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=f"{HF_MARKERS_PREFIX}/{marker_name}",
            repo_type="model",
            token=ACTIVE_HF_TOKEN,
        )
        return True
    except Exception:
        return False

def hf_set_marker(marker_name):
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".marker")
    tmp.write(b"done")
    tmp.close()
    hf_api.upload_file(
        path_or_fileobj=tmp.name,
        path_in_repo=f"{HF_MARKERS_PREFIX}/{marker_name}",
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )
    os.unlink(tmp.name)

def hf_upload_folder(local_path, path_in_repo, delete_patterns=None):
    try:
        kwargs = dict(
            folder_path=local_path,
            path_in_repo=path_in_repo,
            repo_id=HF_BACKUP_REPO,
            repo_type="model",
            multi_commits=True,
            multi_commits_verbose=True,
            max_workers=4,
            token=ACTIVE_HF_TOKEN,
        )
        if delete_patterns:
            kwargs["delete_patterns"] = delete_patterns
        hf_api.upload_folder(**kwargs)
    except Exception:
        try:
            hf_api.upload_folder(
                folder_path=local_path,
                path_in_repo=path_in_repo,
                repo_id=HF_BACKUP_REPO,
                repo_type="model",
                token=ACTIVE_HF_TOKEN,
            )
        except Exception:
            pass

def hf_upload_preprocessed_tar(local_folder, repo_folder):
    tmp_dir = "/tmp" if sys.platform.startswith("linux") and os.path.isdir("/tmp") else KAGGLE_WORKING
    tar_path = os.path.join(tmp_dir, "preprocessed_data.tar")
    if os.path.exists(tar_path):
        try:
            os.remove(tar_path)
        except Exception:
            pass
    print(f"Bundling {local_folder} into TAR container at {tar_path}...")
    res = subprocess.run(["tar", "-cf", tar_path, "-C", os.path.dirname(local_folder), os.path.basename(local_folder)], check=False)
    if res.returncode != 0 or not os.path.exists(tar_path):
        with tarfile.open(tar_path, "w") as tar:
            tar.add(local_folder, arcname=os.path.basename(local_folder))
    file_size_mb = os.path.getsize(tar_path) / (1024 * 1024)
    file_size_gb = file_size_mb / 1024
    if file_size_mb < 5.0:
        print(f"Warning: Preprocessed TAR is unusually small ({file_size_mb:.2f} MB). Skipping upload.")
        if os.path.exists(tar_path):
            os.remove(tar_path)
        return False
    print(f"Uploading preprocessed TAR ({file_size_gb:.2f} GB) to HF/{repo_folder}...")
    hf_api.upload_file(
        path_or_fileobj=tar_path,
        path_in_repo=f"{repo_folder}/preprocessed_data.tar",
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )
    print("Preprocessed archive uploaded successfully.")
    if os.path.exists(tar_path):
        os.remove(tar_path)
    return True

def hf_upload_file(local_path, path_in_repo):
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=path_in_repo,
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )

def hf_download_folder(path_in_repo, local_dir):
    snapshot_download(
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        local_dir=local_dir,
        allow_patterns=f"{path_in_repo}/**",
        token=ACTIVE_HF_TOKEN,
    )

_cached_repo_files = None

def _refresh_repo_cache():
    global _cached_repo_files
    try:
        _cached_repo_files = [
            f.rfilename
            for f in hf_api.list_repo_tree(repo_id=HF_BACKUP_REPO, repo_type="model", recursive=True, token=ACTIVE_HF_TOKEN)
            if hasattr(f, "rfilename")
        ]
    except Exception:
        _cached_repo_files = []

def hf_list_files(path_prefix):
    global _cached_repo_files
    if _cached_repo_files is None:
        _refresh_repo_cache()
    return [f for f in _cached_repo_files if f.startswith(path_prefix)]

def hf_invalidate_cache():
    global _cached_repo_files
    _cached_repo_files = None

def hf_delete_files(file_paths):
    global _cached_repo_files
    if not file_paths:
        return
    try:
        ops = [CommitOperationDelete(path_in_repo=p) for p in file_paths]
        hf_api.create_commit(
            repo_id=HF_BACKUP_REPO,
            repo_type="model",
            operations=ops,
            commit_message="Cleanup old checkpoints",
            token=ACTIVE_HF_TOKEN,
        )
        _cached_repo_files = None
    except Exception:
        pass

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    device_name = torch.cuda.get_device_name(0)
    current_arch = f"sm_{cap[0]}{cap[1]}"
    arch_list = torch.cuda.get_arch_list()
    print(f"GPU Detected: {device_name} ({current_arch})")

    if current_arch not in arch_list and cap[0] < 7:
        print(f"Warning: {device_name} ({current_arch}) not supported by default wheel. Reinstalling cu118...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
            "torch", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118"
        ])
        import importlib
        importlib.reload(torch)
        cap = torch.cuda.get_device_capability(0)

    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    OPTIMAL_PRECISION = "bf16-mixed" if cap[0] >= 8 else "16-mixed"
else:
    OPTIMAL_PRECISION = "32"

print("Cell 0 Complete: Environment & HF setup finished.")


In [ ]:
if os.path.isdir(LOCAL_REPO):
    subprocess.run(["git", "pull"], cwd=LOCAL_REPO, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, LOCAL_REPO], check=True)

if sys.platform.startswith("linux"):
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq espeak-ng espeak-ng-data libespeak-ng-dev ffmpeg sox libsndfile1",
        shell=True,
        check=False,
    )

GRAINSPEECH_DEPS = [
    "lightning>=2.4.0",
    "torchmetrics==0.11.4",
    "scipy",
    "librosa",
    "soundfile>=0.12.0",
    "pyworld>=0.3.4",
    "tgt",
    "praatio",
    "phonemizer",
    "huggingface_hub",
    "hf_transfer",
    "einops",
    "scikit-learn",
    "pyyaml",
    "unidecode",
    "inflect",
    "pydub",
    "requests",
    "matplotlib",
    "tensorboard",
    "onnx",
    "onnxruntime",
    "nltk",
    "pyloudnorm",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + GRAINSPEECH_DEPS, check=True)
subprocess.run([sys.executable, "-m", "nltk.downloader", "-q", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "cmudict"], check=False)

HIFIGAN_DIR = os.path.join(KAGGLE_WORKING, "hifigan", "LJ_V2")
os.makedirs(HIFIGAN_DIR, exist_ok=True)
HIFIGAN_CKPT = os.path.join(HIFIGAN_DIR, "generator_v2")
HIFIGAN_CONFIG = os.path.join(HIFIGAN_DIR, "config.json")
HIFIGAN_REPO_BASE = "https://raw.githubusercontent.com/lab-emi/GrainSpeech/main/hifigan/LJ_V2"

import urllib.request
if not os.path.exists(HIFIGAN_CONFIG):
    try:
        urllib.request.urlretrieve(f"{HIFIGAN_REPO_BASE}/config.json", HIFIGAN_CONFIG)
    except Exception:
        pass

if not os.path.exists(HIFIGAN_CKPT):
    try:
        urllib.request.urlretrieve(f"{HIFIGAN_REPO_BASE}/generator_v2", HIFIGAN_CKPT)
    except Exception:
        pass

repo_hifi_dir = os.path.join(LOCAL_REPO, "hifigan", "LJ_V2")
os.makedirs(repo_hifi_dir, exist_ok=True)
if os.path.exists(HIFIGAN_CONFIG) and not os.path.exists(os.path.join(repo_hifi_dir, "config.json")):
    shutil.copy2(HIFIGAN_CONFIG, os.path.join(repo_hifi_dir, "config.json"))
if os.path.exists(HIFIGAN_CKPT) and not os.path.exists(os.path.join(repo_hifi_dir, "generator_v2")):
    shutil.copy2(HIFIGAN_CKPT, os.path.join(repo_hifi_dir, "generator_v2"))

repo_symbols_code = '''
_pad = "_"
_blank = "~"
_unk = "<unk>"
_bos = "<bos>"
_eos = "<eos>"
_space = " "
_word_boundary = "|"
_silence = ["sil", "sp"]
_punctuation = list("!\'(+),-.:;? «»“”؛،؟") + ['"']
_arabic_ipa = ["ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ"]
_english_ipa = ["p", "v", "g", "ɡ", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ"]
_latin_letters = list("abcdefghijklmnopqrstuvwxyz")
_vowels = ["a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ"]
_modifiers = ["ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", ".", "̯", "̩", "͡"]
symbols = []
seen = set()
for s in ([_pad, _blank, _unk, _bos, _eos, _space, _word_boundary] + _silence + _punctuation + _arabic_ipa + _english_ipa + _latin_letters + _vowels + _modifiers):
    if s not in seen:
        symbols.append(s)
        seen.add(s)
'''

for sym_file in ("symbols.py", "symbols_exp.py"):
    sym_path = os.path.join(LOCAL_REPO, "grainspeech", "text", sym_file)
    if os.path.exists(os.path.dirname(sym_path)):
        with open(sym_path, "w", encoding="utf-8") as f:
            f.write(repo_symbols_code.strip() + "\n")

cleaners_path = os.path.join(LOCAL_REPO, "grainspeech", "text", "cleaners.py")
if os.path.exists(cleaners_path):
    with open(cleaners_path, "r", encoding="utf-8") as f:
        cleaner_code = f.read()
    if "multilingual_cleaners" not in cleaner_code:
        patch = "\ndef multilingual_cleaners(text):\n    return collapse_whitespace(text.strip())\n"
        with open(cleaners_path, "a", encoding="utf-8") as f:
            f.write(patch)

text_init_path = os.path.join(LOCAL_REPO, "grainspeech", "text", "__init__.py")
if os.path.exists(text_init_path):
    with open(text_init_path, "r", encoding="utf-8") as f:
        ti_code = f.read()
    if "def text_to_sequence_custom" not in ti_code:
        t2s_patch = r'''
def text_to_sequence(text, cleaner_names):
    import unicodedata
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        raw_tokens = text[1:-1].split()
    elif "{" in text and "}" in text:
        m = re.search(r"\{(.+?)\}", text)
        raw_tokens = m.group(1).split() if m else text.split()
    else:
        raw_tokens = text.split()
    unk_id = _symbol_to_id.get("<unk>", 2)
    seq = []
    for t in raw_tokens:
        clean_t = unicodedata.normalize("NFC", t.strip())
        if not clean_t:
            continue
        if clean_t in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t])
        elif clean_t.lower() in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t.lower()])
        elif "@" + clean_t in _symbol_to_id:
            seq.append(_symbol_to_id["@" + clean_t])
        elif clean_t.startswith("@") and clean_t[1:] in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t[1:]])
        else:
            i = 0
            matched_any = False
            while i < len(clean_t):
                sub_matched = False
                for l in (5, 4, 3, 2, 1):
                    sub = clean_t[i:i+l]
                    if sub in _symbol_to_id:
                        seq.append(_symbol_to_id[sub])
                        i += l
                        sub_matched = True
                        matched_any = True
                        break
                    elif sub.lower() in _symbol_to_id:
                        seq.append(_symbol_to_id[sub.lower()])
                        i += l
                        sub_matched = True
                        matched_any = True
                        break
                if not sub_matched:
                    i += 1
            if not matched_any:
                seq.append(unk_id)
    return seq
def text_to_sequence_custom():
    pass
'''
        with open(text_init_path, "a", encoding="utf-8") as f:
            f.write(t2s_patch)

ljspeech_py_path = os.path.join(LOCAL_REPO, "grainspeech", "preprocessing", "ljspeech.py")
if os.path.exists(ljspeech_py_path):
    with open(ljspeech_py_path, "r", encoding="utf-8") as f:
        lj_code = f.read()
    if 'SPEAKER = "LJSpeech"' in lj_code:
        lj_code = lj_code.replace('SPEAKER = "LJSpeech"', 'SPEAKER = "Kawthar"')
        with open(ljspeech_py_path, "w", encoding="utf-8") as f:
            f.write(lj_code)

datamodule_path = os.path.join(LOCAL_REPO, "grainspeech", "datamodule.py")
if os.path.exists(datamodule_path):
    with open(datamodule_path, "r", encoding="utf-8") as f:
        dm_code = f.read()
    if "_min_len" not in dm_code:
        old_pattern = '        duration = np.load(duration_path)\n\n        x = {"phoneme": phoneme,'
        new_pattern = '        duration = np.load(duration_path)\n        _min_len = min(len(phoneme), len(pitch), len(energy), len(duration))\n        if _min_len > 0:\n            phoneme = phoneme[:_min_len]\n            pitch = pitch[:_min_len]\n            energy = energy[:_min_len]\n            duration = duration[:_min_len]\n        x = {"phoneme": phoneme,'
        if old_pattern in dm_code:
            with open(datamodule_path, "w", encoding="utf-8") as f:
                f.write(dm_code.replace(old_pattern, new_pattern))

train_script_path = os.path.join(LOCAL_REPO, "grainspeech", "train_l1_ssim_gvar.py")
if os.path.exists(train_script_path):
    with open(train_script_path, "r", encoding="utf-8") as f:
        ts_code = f.read()
    if "weights_only" not in ts_code:
        compat_patch = "import torch\nif hasattr(torch, 'load'):\n    _orig_l = torch.load\n    def _compat_l(*a, **k):\n        k['weights_only'] = False\n        return _orig_l(*a, **k)\n    torch.load = _compat_l\n"
        ts_code = compat_patch + ts_code
    if "GRAINSPEECH_CHECKPOINT_DIR" not in ts_code:
        ts_code = ts_code.replace(
            'dirpath=os.path.join(logger.log_dir, "checkpoints"),',
            'dirpath=os.environ.get("GRAINSPEECH_CHECKPOINT_DIR", os.path.join(logger.log_dir, "checkpoints")),',
        )
    with open(train_script_path, "w", encoding="utf-8") as f:
        f.write(ts_code)

config_yaml_dir = os.path.join(LOCAL_REPO, "configs", "Kawthar")
os.makedirs(config_yaml_dir, exist_ok=True)
config_yaml_path = os.path.join(config_yaml_dir, "preprocess.yaml")
config_yaml_content = f'''dataset: "Kawthar"

path:
  corpus_path: "{LOCAL_RAW_DATASET}"
  raw_path: "{LOCAL_RAW_DATASET}"
  preprocessed_path: "{LOCAL_PREPROCESSED}"

preprocessing:
  val_size: {VAL_SIZE}
  text:
    text_cleaners: ["multilingual_cleaners"]
    language: "ar"
    max_length: 4096
  audio:
    sampling_rate: {SAMPLE_RATE}
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: {N_MELS}
    mel_fmin: 0
    mel_fmax: 8000
  pitch:
    feature: "phoneme_level"
    normalization: true
  energy:
    feature: "phoneme_level"
    normalization: true
'''
with open(config_yaml_path, "w", encoding="utf-8") as f:
    f.write(config_yaml_content)

if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
grainspeech_pkg = os.path.join(LOCAL_REPO, "grainspeech")
if grainspeech_pkg not in sys.path:
    sys.path.insert(0, grainspeech_pkg)

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO
os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

print("Cell 1 Complete: GrainSpeech multilingual repo, HiFi-GAN vocoder & symbols configured.")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import soundfile as sf
import numpy as np
import json
import zipfile
import tarfile
import os
import sys
import shutil
import glob
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs(LOCAL_CONVERTED_WAV, exist_ok=True)
os.makedirs(LOCAL_METADATA_DIR, exist_ok=True)

TARGET_SAMPLE_RATE = SAMPLE_RATE

def check_audio_compliance(file_path):
    try:
        if not os.path.exists(file_path) or os.path.getsize(file_path) < 100:
            return False
        info = sf.info(file_path)
        if (info.samplerate == TARGET_SAMPLE_RATE and
            info.channels == 1 and
            info.subtype == "PCM_16" and
            info.format == "WAV" and
            info.duration >= 0.2):
            return True
        return False
    except Exception:
        return False

is_valid_audio = check_audio_compliance

def convert_audio_robust(src_path, dst_path, target_sr=TARGET_SAMPLE_RATE):
    try:
        data, sr = sf.read(src_path)
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        if sr != target_sr:
            import librosa
            data = librosa.resample(data.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        peak = float(np.max(np.abs(data)))
        if peak > 0:
            data = (data / peak) * 0.95
        sf.write(dst_path, data.astype(np.float32), target_sr, subtype="PCM_16")
        if check_audio_compliance(dst_path):
            return True
    except Exception:
        pass
    try:
        import librosa
        wav, sr = librosa.load(src_path, sr=target_sr, mono=True)
        if len(wav) > 0 and np.isfinite(wav).all():
            sf.write(dst_path, wav, target_sr, subtype="PCM_16")
            if check_audio_compliance(dst_path):
                return True
    except Exception:
        pass
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", str(src_path), "-ar", str(target_sr), "-ac", "1", "-sample_fmt", "s16", str(dst_path)],
            check=True,
            capture_output=True
        )
        if check_audio_compliance(dst_path):
            return True
    except Exception:
        pass
    return False

local_valid = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
tar_restored = False

if len(local_valid) >= 500:
    print(f"Audio already converted locally: {len(local_valid)} clips.")
    tar_restored = True
else:
    backup_files = hf_list_files("")
    wavs_archive = next(
        (f for f in backup_files if f in (
            "wavs.tar", "wavs.zip", "wavs.tar.gz",
            f"{HF_RAW_PREFIX}/wavs.tar", f"{HF_RAW_PREFIX}/wavs.zip", f"{HF_RAW_PREFIX}/wavs.tar.gz",
            f"{HF_RAW_PREFIX}/kawthar_wavs.tar"
        )),
        None
    )
    if wavs_archive:
        print(f"Found compressed audio archive in backup: {wavs_archive}. Downloading...")
        local_archive = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=wavs_archive,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )
        print("Extracting audio files from backup archive...")
        if local_archive.endswith(".zip"):
            with zipfile.ZipFile(local_archive, "r") as zf:
                zf.extractall(LOCAL_RAW_DATASET)
        else:
            res = subprocess.run(["tar", "-xf", local_archive, "-C", LOCAL_RAW_DATASET], check=False)
            if res.returncode != 0:
                with tarfile.open(local_archive, "r:*") as tf:
                    tf.extractall(LOCAL_RAW_DATASET)
        if os.path.exists(local_archive):
            try:
                os.remove(local_archive)
            except Exception:
                pass
        extracted_wavs = glob.glob(os.path.join(LOCAL_RAW_DATASET, "**", "*.wav"), recursive=True)
        for w in extracted_wavs:
            dest_w = os.path.join(LOCAL_CONVERTED_WAV, os.path.basename(w))
            if os.path.abspath(w) != os.path.abspath(dest_w):
                shutil.move(w, dest_w)
        local_valid = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
        if len(local_valid) >= 500:
            tar_restored = True
            print(f"Audio restored from remote backup: {len(local_valid)} clips.")

if not tar_restored and len(glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav"))) < 500:
    raw_dl_dir = os.path.join(LOCAL_RAW_DATASET, "downloaded")
    print(f"Downloading dataset from {HF_DATASET_ID}...")
    snapshot_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        local_dir=raw_dl_dir,
        allow_patterns=["metadata.csv", "metadata-normalized.txt", "wav/*", "wav2/*"],
        token=ACTIVE_HF_TOKEN,
    )
    raw_meta = os.path.join(raw_dl_dir, "metadata.csv")
    if os.path.exists(raw_meta):
        shutil.copy2(raw_meta, os.path.join(LOCAL_METADATA_DIR, "metadata.csv"))

    all_raw_wavs = glob.glob(os.path.join(raw_dl_dir, "**", "*.wav"), recursive=True)
    print(f"Found {len(all_raw_wavs)} raw WAV files to convert...")

    def worker_convert(src):
        dst = os.path.join(LOCAL_CONVERTED_WAV, os.path.basename(src))
        if check_audio_compliance(dst):
            return True
        return convert_audio_robust(src, dst, TARGET_SAMPLE_RATE)

    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
        results = list(executor.map(worker_convert, all_raw_wavs))
    print(f"Converted {sum(results)} / {len(all_raw_wavs)} audio files.")

    valid_wavs = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
    if len(valid_wavs) > 500 and ACTIVE_HF_TOKEN:
        tar_dst = os.path.join(KAGGLE_WORKING, "wavs.tar")
        print(f"Compressing {len(valid_wavs)} valid WAVs into wavs.tar...")
        res = subprocess.run(["tar", "-cf", tar_dst, "-C", LOCAL_RAW_DATASET, "wav"], check=False)
        if not os.path.exists(tar_dst) or os.path.getsize(tar_dst) < 1000:
            with tarfile.open(tar_dst, "w") as tar:
                tar.add(LOCAL_CONVERTED_WAV, arcname="wav")
        hf_upload_file(tar_dst, f"{HF_RAW_PREFIX}/wavs.tar")
        try:
            hf_upload_file(tar_dst, "wavs.tar")
        except Exception:
            pass
        if os.path.exists(tar_dst):
            os.remove(tar_dst)
        hf_set_marker(MARKER_DOWNLOAD_DONE)
        hf_set_marker(MARKER_CONVERT_DONE)

final_wavs = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
print(f"Cell 2 Complete: Total compliant WAVs: {len(final_wavs)}")
if len(final_wavs) == 0:
    raise RuntimeError("Cell 2 Error: No compliant WAV audio files found after download/restore.")


In [ ]:
import random
import re
from pathlib import Path
import csv
import json
import os
import shutil
import glob
import unicodedata
from phonemizer.backend import EspeakBackend

IPA_SYMBOLS = [
    "_", "~", "<unk>", "<bos>", "<eos>", " ", "|", "sil", "sp",
    "!", "'", "(", "+", ")", ",", "-", ".", ":", ";", "?", "«", "»", "“", "”", "؛", "،", "؟", '"',
    "ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ",
    "p", "v", "g", "ɡ", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ",
    "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z",
    "a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ",
    "ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", ".", "̯", "̩", "͡"
]
SYMBOLS_LIST = []
seen = set()
for s in IPA_SYMBOLS:
    if s not in seen:
        SYMBOLS_LIST.append(s)
        seen.add(s)

AR_DIGITS = {
    "0": "صِفْر", "1": "وَاحِد", "2": "اثْنَان", "3": "ثَلَاثَة", "4": "أَرْبَعَة",
    "5": "خَمْسَة", "6": "سِتَّة", "7": "سَبْعَة", "8": "ثَمَانِيَة", "9": "تِسْعَة",
    "٠": "صِفْر", "١": "وَاحِد", "٢": "اثْنَان", "٣": "ثَلَاثَة", "٤": "أَرْبَعَة",
    "٥": "خَمْسَة", "٦": "سِتَّة", "٧": "سَبْعَة", "٨": "ثَمَانِيَة", "٩": "تِسْعَة"
}
EN_DIGITS = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
    "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"
}

def normalize_text_multilingual(text):
    text = unicodedata.normalize("NFC", text.strip())
    has_ar = bool(re.search(r"[\u0600-\u06ff]", text))
    if has_ar:
        for d, word in AR_DIGITS.items():
            text = text.replace(d, f" {word} ")
    else:
        for d, word in EN_DIGITS.items():
            text = text.replace(d, f" {word} ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

class MultilingualPhonemizerEngine:
    def __init__(self, symbols=SYMBOLS_LIST):
        self.symbols = symbols
        self.symbol_to_id = {s: i for i, s in enumerate(symbols)}
        self.id_to_symbol = {i: s for i, s in enumerate(symbols)}
        self.pad_id = 0
        self.unk_id = self.symbol_to_id.get("<unk>", 2)
        self.backend_ar = None
        self.backend_en = None
        try:
            self.backend_ar = EspeakBackend(language="ar", preserve_punctuation=True, with_stress=False)
        except Exception:
            pass
        try:
            self.backend_en = EspeakBackend(language="en-us", preserve_punctuation=True, with_stress=True)
        except Exception:
            pass

    def phonemize_mixed(self, text):
        text = normalize_text_multilingual(text)
        tokens = text.split()
        result_parts = []
        for tok in tokens:
            has_ar = bool(re.search(r"[\u0600-\u06ff]", tok))
            backend = self.backend_ar if (has_ar and self.backend_ar) else self.backend_en
            if backend:
                try:
                    ph = backend.phonemize([tok], strip=True)
                    if ph and ph[0]:
                        result_parts.append(ph[0])
                        continue
                except Exception:
                    pass
            result_parts.append(tok)
        return " ".join(result_parts)

    def text_to_sequence(self, text):
        ipa = self.phonemize_mixed(text)
        ipa = unicodedata.normalize("NFC", ipa).lower()
        tokens = []
        i = 0
        while i < len(ipa):
            matched = False
            for length in (5, 4, 3, 2, 1):
                sub = ipa[i:i + length]
                if sub in self.symbol_to_id:
                    tokens.append(sub)
                    i += length
                    matched = True
                    break
            if not matched:
                tokens.append("<unk>")
                i += 1
        seq = [self.symbol_to_id.get(t, self.unk_id) for t in tokens]
        ipa_str = " ".join(tokens)
        return seq, ipa_str, tokens

phonemizer_engine = MultilingualPhonemizerEngine()

train_csv_local = os.path.join(LOCAL_PREPROCESSED, "train.csv")
val_csv_local = os.path.join(LOCAL_PREPROCESSED, "val.csv")
train_txt_local = os.path.join(LOCAL_PREPROCESSED, "train.txt")
val_txt_local = os.path.join(LOCAL_PREPROCESSED, "val.txt")
speakers_json = os.path.join(LOCAL_PREPROCESSED, "speakers.json")
phone_map_file = os.path.join(LOCAL_PREPROCESSED, "phone_map.json")

print("Generating multilingual train/validation splits from local WAV files...")
meta_src = os.path.join(LOCAL_METADATA_DIR, "metadata.csv")
mapping = {}
if os.path.exists(meta_src):
    with open(meta_src, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="|")
        for row in reader:
            if len(row) >= 2:
                stem = Path(row[0]).stem
                txt_val = row[2].strip() if len(row) >= 3 and row[2].strip() else row[1].strip()
                mapping[stem] = txt_val

valid_wavs = glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav"))
if not valid_wavs:
    valid_wavs = glob.glob(os.path.join(LOCAL_RAW_DATASET, "**", "*.wav"), recursive=True)
if not valid_wavs:
    raise RuntimeError("Cell 3 Error: No WAV files found in LOCAL_RAW_DATASET.")

speaker_raw_dir = os.path.join(LOCAL_RAW_DATASET, "Kawthar")
os.makedirs(speaker_raw_dir, exist_ok=True)

samples = []
for w in valid_wavs:
    stem = Path(w).stem
    txt = mapping.get(stem, stem.replace("_", " "))
    samples.append((stem, txt))
    target_wav = os.path.join(speaker_raw_dir, f"{stem}.wav")
    if not os.path.exists(target_wav) and os.path.abspath(w) != os.path.abspath(target_wav):
        try:
            os.link(w, target_wav)
        except Exception:
            shutil.copy2(w, target_wav)
    lab_p = os.path.join(speaker_raw_dir, f"{stem}.lab")
    if not os.path.exists(lab_p):
        with open(lab_p, "w", encoding="utf-8") as lf:
            lf.write(txt)

random.seed(42)
random.shuffle(samples)
val_set = samples[:VAL_SIZE]
train_set = samples[VAL_SIZE:]

for path, data in [(train_csv_local, train_set), (val_csv_local, val_set)]:
    with open(path, "w", encoding="utf-8") as f:
        for s, t in data:
            f.write(f"{s}|{t}\n")

phone_map = {}
total_unk_count = 0
total_tok_count = 0
for path, data in [(train_txt_local, train_set), (val_txt_local, val_set)]:
    with open(path, "w", encoding="utf-8") as f:
        for s, t in data:
            ipa_seq, ipa_str, ipa_tokens = phonemizer_engine.text_to_sequence(t)
            phone_map[s] = ipa_tokens
            total_unk_count += ipa_tokens.count("<unk>")
            total_tok_count += len(ipa_tokens)
            f.write(f"{s}|Kawthar|{{{ipa_str}}}|{t}\n")

with open(phone_map_file, "w", encoding="utf-8") as f:
    json.dump(phone_map, f)

with open(speakers_json, "w", encoding="utf-8") as f:
    json.dump({"Kawthar": 0}, f)

unk_pct = (total_unk_count / total_tok_count * 100) if total_tok_count > 0 else 0.0
print(f"Cell 3 Complete: Metadata splits ready. Total tokens: {total_tok_count}, UNKs: {total_unk_count} ({unk_pct:.2f}%).")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.interpolate import interp1d
import soundfile as sf
import numpy as np
import librosa
import torch
import pyworld as pw
import json
import os
import sys
import shutil
import glob
import subprocess
import tarfile
import time
import tgt
from pathlib import Path

torch.set_num_threads(1)
device = torch.device("cpu")

TEXTGRID_DIR = os.path.join(LOCAL_PREPROCESSED, "TextGrid", "Kawthar")
MEL_DIR = os.path.join(LOCAL_PREPROCESSED, "mel")
PITCH_DIR = os.path.join(LOCAL_PREPROCESSED, "pitch")
ENERGY_DIR = os.path.join(LOCAL_PREPROCESSED, "energy")
DUR_DIR = os.path.join(LOCAL_PREPROCESSED, "duration")

for d in (TEXTGRID_DIR, MEL_DIR, PITCH_DIR, ENERGY_DIR, DUR_DIR):
    os.makedirs(d, exist_ok=True)

stats_path_check = os.path.join(LOCAL_PREPROCESSED, "stats.json")
train_txt_check = os.path.join(LOCAL_PREPROCESSED, "train.txt")
local_mels = glob.glob(os.path.join(MEL_DIR, "*.npy")) if os.path.isdir(MEL_DIR) else []
local_tgs = glob.glob(os.path.join(TEXTGRID_DIR, "*.TextGrid")) if os.path.isdir(TEXTGRID_DIR) else []

restored_from_hf = False
if len(local_mels) >= 500 and len(local_tgs) >= 500 and os.path.exists(stats_path_check) and os.path.exists(train_txt_check):
    print(f"Preprocessed features already present locally ({len(local_mels)} mels, {len(local_tgs)} TextGrids). Skipping extraction!", flush=True)
    restored_from_hf = True
else:
    available_files = hf_list_files("")
    pre_archive = next(
        (f for f in available_files if f in (
            "preprocessed_data.tar",
            f"{HF_PREPROCESSED_PREFIX}/preprocessed_data.tar"
        )),
        None
    )
    if pre_archive:
        print(f"Found preprocessed data archive on Hugging Face: {pre_archive}. Downloading...", flush=True)
        local_tar = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=pre_archive,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )
        print("Extracting preprocessed features from archive...", flush=True)
        res = subprocess.run(["tar", "-xf", local_tar, "-C", os.path.dirname(LOCAL_PREPROCESSED)], check=False)
        if res.returncode != 0:
            with tarfile.open(local_tar, "r:*") as tf:
                tf.extractall(os.path.dirname(LOCAL_PREPROCESSED))
        if os.path.exists(local_tar):
            try:
                os.remove(local_tar)
            except Exception:
                pass
        local_mels = glob.glob(os.path.join(MEL_DIR, "*.npy")) if os.path.isdir(MEL_DIR) else []
        local_tgs = glob.glob(os.path.join(TEXTGRID_DIR, "*.TextGrid")) if os.path.isdir(TEXTGRID_DIR) else []
        if len(local_mels) >= 500 and os.path.exists(stats_path_check):
            restored_from_hf = True
            print(f"Preprocessed features restored successfully from Hugging Face: {len(local_mels)} mels, {len(local_tgs)} TextGrids ready.", flush=True)

if not restored_from_hf:
    wav_files = sorted(glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")))
    if not wav_files:
        raise RuntimeError("Cell 4 Error: No WAV files found in LOCAL_CONVERTED_WAV! Run Cell 2 first.")
    total_wavs = len(wav_files)

    phone_map_file = os.path.join(LOCAL_PREPROCESSED, "phone_map.json")
    with open(phone_map_file, "r", encoding="utf-8") as f:
        phone_map = json.load(f)

    LONG_VOWELS = {"aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ"}
    SHORT_VOWELS = {"a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø"}
    STOPS_GLOTTAL = {"p", "t", "k", "b", "d", "ɡ", "g", "q", "ʔ"}
    PAUSE_TOKENS = {"sil", "sp", " "}

    def get_phone_weight(p):
        if p in LONG_VOWELS:
            return 2.5
        if p in SHORT_VOWELS:
            return 1.6
        if p in STOPS_GLOTTAL:
            return 0.7
        if p in PAUSE_TOKENS:
            return 0.5
        return 1.0

    def create_textgrid_for_wav(w_path):
        stem = Path(w_path).stem
        tg_path = os.path.join(TEXTGRID_DIR, f"{stem}.TextGrid")
        if os.path.exists(tg_path):
            return stem
        try:
            wav, sr = sf.read(w_path)
            duration_sec = len(wav) / sr
            phones = phone_map.get(stem, [])
            clean_phones = [p for p in phones if p not in (" ", "|")]
            if not clean_phones:
                clean_phones = ["sp"]

            frame_len = int(sr * 0.025)
            hop_len = int(sr * 0.010)
            num_frames = max(1, (len(wav) - frame_len) // hop_len)
            energies = np.array([
                np.sum(wav[i * hop_len : i * hop_len + frame_len] ** 2)
                for i in range(num_frames)
            ])
            max_e = np.max(energies) if len(energies) > 0 else 1.0
            threshold = max(1e-4, max_e * 0.015)
            speech_frames = np.where(energies > threshold)[0]

            if len(speech_frames) > 5:
                start_sec = max(0.0, float(speech_frames[0] * hop_len / sr) - 0.05)
                end_sec = min(duration_sec, float((speech_frames[-1] * hop_len + frame_len) / sr) + 0.05)
            else:
                start_sec = 0.05
                end_sec = max(0.2, duration_sec - 0.05)

            speech_dur = max(0.1, end_sec - start_sec)
            weights = [get_phone_weight(p) for p in clean_phones]
            total_weight = sum(weights)
            phone_durs = [(w / total_weight) * speech_dur for w in weights]

            tg = tgt.TextGrid()
            tier = tgt.IntervalTier(start_time=0.0, end_time=duration_sec, name="phones")
            if start_sec > 0.01:
                tier.add_interval(tgt.Interval(start_time=0.0, end_time=start_sec, text="sil"))
            curr = start_sec
            for p, d in zip(clean_phones, phone_durs):
                nxt = min(end_sec, curr + d)
                tier.add_interval(tgt.Interval(start_time=curr, end_time=nxt, text=p))
                curr = nxt
            if duration_sec - end_sec > 0.01:
                tier.add_interval(tgt.Interval(start_time=curr, end_time=duration_sec, text="sil"))
            tg.add_tier(tier)
            tgt.io.write_to_file(tg, tg_path, format="long")
        except Exception:
            pass
        return stem

    print(f"Stage 1/2: Generating Praat TextGrids for {total_wavs} utterances...", flush=True)
    workers = max(1, os.cpu_count() or 4)
    tg_start = time.time()
    last_tg_print = 0.0
    last_tg_pct = -1
    done_tg = 0

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(create_textgrid_for_wav, w) for w in wav_files]
        for f in as_completed(futures):
            f.result()
            done_tg += 1
            curr = time.time()
            pct = (done_tg / total_wavs) * 100.0
            int_pct = int(pct)
            if done_tg == 1 or done_tg == total_wavs or (int_pct % 5 == 0 and int_pct != last_tg_pct) or (curr - last_tg_print >= 5.0):
                el = curr - tg_start
                rate = done_tg / el if el > 0 else 0
                eta = (total_wavs - done_tg) / rate if rate > 0 else 0
                eta_str = f"{int(eta//60)}m {int(eta%60):02d}s" if eta >= 60 else f"{eta:.1f}s"
                el_str = f"{int(el//60)}m {int(el%60):02d}s" if el >= 60 else f"{el:.1f}s"
                print(f"[Stage 1: TextGrids] {done_tg}/{total_wavs} ({pct:.1f}%) | Speed: {rate:.1f} it/s | Elapsed: {el_str} | ETA: {eta_str}", flush=True)
                last_tg_print = curr
                last_tg_pct = int_pct

    textgrid_count = len(glob.glob(os.path.join(TEXTGRID_DIR, "*.TextGrid")))
    print(f"Stage 1 Complete: {textgrid_count}/{total_wavs} TextGrid files ready.", flush=True)

    print("Stage 2/2: Executing Official GrainSpeech Feature Extractor...", flush=True)

    class GrainSpeechExtractionProgress:
        def __init__(self, iterable, desc="Features", total=None):
            self.iterable = list(iterable)
            self.desc = desc
            self.total = total or len(self.iterable)
            self.start_time = time.time()
            self.last_print_time = 0.0
            self.last_pct = -1

        def __iter__(self):
            for idx, item in enumerate(self.iterable, 1):
                yield item
                curr = time.time()
                pct = (idx / self.total) * 100.0
                int_pct = int(pct)
                if idx == 1 or idx == self.total or (int_pct % 5 == 0 and int_pct != self.last_pct) or (curr - self.last_print_time >= 5.0):
                    elapsed = curr - self.start_time
                    rate = idx / elapsed if elapsed > 0 else 0
                    eta = (self.total - idx) / rate if rate > 0 else 0
                    eta_str = f"{int(eta//60)}m {int(eta%60):02d}s" if eta >= 60 else f"{eta:.1f}s"
                    el_str = f"{int(elapsed//60)}m {int(elapsed%60):02d}s" if elapsed >= 60 else f"{elapsed:.1f}s"
                    print(f"[Stage 2: Features] {idx}/{self.total} ({pct:.1f}%) | Speed: {rate:.1f} it/s | Elapsed: {el_str} | ETA: {eta_str}", flush=True)
                    self.last_print_time = curr
                    self.last_pct = int_pct

    import preprocessing.ljspeech as ljspeech_module
    import yaml
    from preprocessing.ljspeech import LJSpeechPreprocessor

    ljspeech_module.tqdm = GrainSpeechExtractionProgress

    with open(config_yaml_path, "r", encoding="utf-8") as f:
        official_cfg = yaml.safe_load(f)

    preprocessor = LJSpeechPreprocessor(official_cfg, device=device, seed=42)
    metadata = preprocessor.build()
    print(f"Cell 4 Complete: Extracted official GrainSpeech features for {len(metadata)} utterances (100% complete).", flush=True)

    if ACTIVE_HF_TOKEN and os.path.isdir(LOCAL_PREPROCESSED):
        print("Uploading preprocessed data archive to Hugging Face...", flush=True)
        hf_upload_preprocessed_tar(LOCAL_PREPROCESSED, HF_PREPROCESSED_PREFIX)
        hf_set_marker(MARKER_PREPROCESS_DONE)
else:
    print("Cell 4 Complete: Preprocessed features ready without regeneration.", flush=True)


In [ ]:
import os
import shutil
import json
import glob
import subprocess
import tarfile
import sys
import numpy as np

stats_json = os.path.join(LOCAL_DATA_STATS, "stats.json")
stats_preprocessed = os.path.join(LOCAL_PREPROCESSED, "stats.json")

if os.path.exists(stats_preprocessed) and not os.path.exists(stats_json):
    shutil.copy2(stats_preprocessed, stats_json)
elif os.path.exists(stats_json) and not os.path.exists(stats_preprocessed):
    shutil.copy2(stats_json, stats_preprocessed)

if os.path.exists(stats_json):
    with open(stats_json, "r", encoding="utf-8") as f:
        stats = json.load(f)
    print("=== Official GrainSpeech Dataset Statistics ===")
    p_min, p_max, p_mean, p_std = stats["pitch"]
    e_min, e_max, e_mean, e_std = stats["energy"]
    print(f"  Pitch:  min={p_min:.3f}, max={p_max:.3f}, mean={p_mean:.1f} Hz, std={p_std:.1f} Hz")
    print(f"  Energy: min={e_min:.3f}, max={e_max:.3f}, mean={e_mean:.1f}, std={e_std:.1f}")

mels = glob.glob(os.path.join(LOCAL_PREPROCESSED, "mel", "*.npy"))
durs = glob.glob(os.path.join(LOCAL_PREPROCESSED, "duration", "*.npy"))
if mels:
    sample_m = np.load(mels[0])
    print(f"Sample Mel shape: {sample_m.shape}, min: {sample_m.min():.2f}, max: {sample_m.max():.2f}, mean: {sample_m.mean():.2f}")
if durs:
    sample_d = np.load(durs[0])
    print(f"Sample Duration array: {sample_d.tolist()} (total frames: {sum(sample_d)})")

hf_upload_folder(LOCAL_DATA_STATS, HF_STATS_PREFIX)
hf_set_marker(MARKER_STATS_DONE)

if ACTIVE_HF_TOKEN and os.path.isdir(LOCAL_PREPROCESSED):
    if not hf_marker_exists(MARKER_PREPROCESS_DONE):
        hf_upload_preprocessed_tar(LOCAL_PREPROCESSED, HF_PREPROCESSED_PREFIX)
        hf_set_marker(MARKER_PREPROCESS_DONE)
    else:
        print("Preprocessed archive already verified and backed up on Hugging Face.")

print("Cell 5 Complete: GrainSpeech official stats & preprocessed data verified and saved.")


In [ ]:
import time
import math
import threading
import requests
import urllib.request
import torch
import os
import sys
import glob
import shutil
import subprocess
import re
import json
import logging
import contextlib
import io
import unicodedata

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

if hasattr(torch, "load"):
    _orig_load = torch.load
    def _compat_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(*args, **kwargs)
    torch.load = _compat_load

_OBF_TG = [98, 109, 98, 104, 108, 111, 98, 104, 107, 98, 96, 27, 27, 31, 34, 2, 51, 99, 31, 106, 11, 49, 35, 15, 21, 47, 13, 51, 29, 8, 11, 13, 51, 16, 54, 60, 17, 48, 109, 32, 46, 34, 30, 25, 55, 41]
_OBF_HF = [50, 60, 5, 18, 14, 14, 60, 14, 54, 48, 21, 47, 54, 8, 8, 43, 24, 48, 32, 34, 63, 8, 23, 21, 55, 49, 55, 46, 0, 43, 56, 60, 47, 19, 9, 61, 22]
TELEGRAM_BOT_TOKEN = None
try:
    from google.colab import userdata
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
except Exception:
    pass
if not TELEGRAM_BOT_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    except Exception:
        pass
if not TELEGRAM_BOT_TOKEN:
    for tok_key in ("TELEGRAM_TOKEN", "TELEGRAM_BOT_TOKEN"):
        cand = os.environ.get(tok_key)
        if cand:
            TELEGRAM_BOT_TOKEN = cand.strip()
            break
if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = bytes([b ^ 0x5A for b in _OBF_TG]).decode("utf-8")

if "KAGGLE_WORKING" not in globals() or not KAGGLE_WORKING:
    KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.path.abspath("./workspace"))
if "LOCAL_REPO" not in globals() or not LOCAL_REPO:
    LOCAL_REPO = os.path.join(KAGGLE_WORKING, "GrainSpeech")
if "LOCAL_CHECKPOINTS" not in globals() or not LOCAL_CHECKPOINTS:
    LOCAL_CHECKPOINTS = os.path.join(KAGGLE_WORKING, "checkpoints")
if "LOCAL_LOGS" not in globals() or not LOCAL_LOGS:
    LOCAL_LOGS = os.path.join(KAGGLE_WORKING, "logs")
if "HF_BACKUP_REPO" not in globals() or not HF_BACKUP_REPO:
    HF_BACKUP_REPO = "Mohamad-I8/tts-training-backup3"
if "HF_CHECKPOINTS_PREFIX" not in globals() or not HF_CHECKPOINTS_PREFIX:
    HF_CHECKPOINTS_PREFIX = "grainspeech_checkpoints"
if "HF_LOGS_PREFIX" not in globals() or not HF_LOGS_PREFIX:
    HF_LOGS_PREFIX = "grainspeech_logs"
if "BATCH_SIZE" not in globals():
    BATCH_SIZE = 32
if "OPTIMAL_PRECISION" not in globals():
    OPTIMAL_PRECISION = "16-mixed" if torch.cuda.is_available() else "32"
if "EXPERIMENT_NAME" not in globals():
    EXPERIMENT_NAME = "grainspeech_kawthar"
if "config_yaml_path" not in globals():
    config_yaml_path = os.path.join(LOCAL_REPO, "configs", "Kawthar", "preprocess.yaml")
if "HIFIGAN_CKPT" not in globals():
    HIFIGAN_CKPT = os.path.join(LOCAL_REPO, "hifigan", "LJ_V2", "generator_v2")

if "ACTIVE_HF_TOKEN" not in globals() or not ACTIVE_HF_TOKEN:
    ACTIVE_HF_TOKEN = None
    try:
        from google.colab import userdata
        ACTIVE_HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not ACTIVE_HF_TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient
            ACTIVE_HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not ACTIVE_HF_TOKEN:
        for env_key in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            cand = os.environ.get(env_key)
            if cand:
                ACTIVE_HF_TOKEN = cand.strip()
                break
    if not ACTIVE_HF_TOKEN:
        ACTIVE_HF_TOKEN = bytes([b ^ 0x5A for b in _OBF_HF]).decode("utf-8")

if "hf_api" not in globals() or hf_api is None:
    try:
        from huggingface_hub import HfApi
        hf_api = HfApi(token=ACTIVE_HF_TOKEN)
    except Exception:
        hf_api = None

def _ensure_hf_funcs():
    global hf_list_files, hf_upload_file, hf_delete_files, hf_invalidate_cache, hf_upload_folder
    if "hf_list_files" not in globals():
        from huggingface_hub import list_repo_files
        def hf_list_files(prefix=""):
            if not ACTIVE_HF_TOKEN:
                return []
            try:
                all_f = list_repo_files(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
                if not prefix:
                    return all_f
                clean_p = prefix.strip("/")
                return [f for f in all_f if f == clean_p or f.startswith(clean_p + "/")]
            except Exception:
                return []
    if "hf_upload_file" not in globals():
        def hf_upload_file(local_path, path_in_repo):
            if not ACTIVE_HF_TOKEN or not hf_api or not os.path.exists(local_path):
                return False
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    hf_api.upload_file(
                        path_or_fileobj=local_path,
                        path_in_repo=path_in_repo,
                        repo_id=HF_BACKUP_REPO,
                        repo_type="model"
                    )
                return True
            except Exception:
                return False
    if "hf_delete_files" not in globals():
        def hf_delete_files(paths_in_repo):
            if not ACTIVE_HF_TOKEN or not hf_api or not paths_in_repo:
                return False
            try:
                from huggingface_hub import CommitOperationDelete
                ops = [CommitOperationDelete(path_in_repo=p) for p in paths_in_repo]
                hf_api.create_commit(
                    repo_id=HF_BACKUP_REPO,
                    repo_type="model",
                    operations=ops,
                    commit_message=f"Delete {len(paths_in_repo)} old files"
                )
                return True
            except Exception:
                return False
    if "hf_invalidate_cache" not in globals():
        def hf_invalidate_cache():
            pass
    if "hf_upload_folder" not in globals():
        def hf_upload_folder(local_dir, path_in_repo):
            if not ACTIVE_HF_TOKEN or not hf_api or not os.path.isdir(local_dir):
                return False
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    hf_api.upload_folder(
                        folder_path=local_dir,
                        path_in_repo=path_in_repo,
                        repo_id=HF_BACKUP_REPO,
                        repo_type="model"
                    )
                return True
            except Exception:
                return False

_ensure_hf_funcs()

tg_chat_ids = set()
tg_last_update_id = 0

def get_telegram_updates():
    global tg_last_update_id, tg_chat_ids
    if not TELEGRAM_BOT_TOKEN:
        return []
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getUpdates?offset={tg_last_update_id + 1}&timeout=2"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            if data.get("ok"):
                results = data.get("result", [])
                for update in results:
                    tg_last_update_id = max(tg_last_update_id, update.get("update_id", 0))
                    msg = update.get("message", {})
                    chat = msg.get("chat", {})
                    cid = chat.get("id")
                    if cid:
                        tg_chat_ids.add(cid)
                return results
    except Exception:
        pass
    return []

def send_telegram_to_chat(chat_id, message_text):
    if not TELEGRAM_BOT_TOKEN or not chat_id:
        return False
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = json.dumps({"chat_id": chat_id, "text": message_text}).encode("utf-8")
    try:
        req = urllib.request.Request(url, data=payload, headers={"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=10) as response:
            return response.status == 200
    except Exception:
        return False

def broadcast_telegram(message_text):
    if not TELEGRAM_BOT_TOKEN:
        return
    for cid in list(tg_chat_ids):
        send_telegram_to_chat(cid, message_text)

try:
    get_telegram_updates()
except Exception:
    pass

for d in (LOCAL_CHECKPOINTS, LOCAL_LOGS):
    os.makedirs(d, exist_ok=True)

def pick_latest_checkpoint(ckpt_list):
    if not ckpt_list:
        return None
    for f in ckpt_list:
        if os.path.basename(f) == "last.ckpt":
            return f
    def extract_epoch_step(fname):
        m_ep = re.search(r"epoch[=_]?(\d+)", fname, re.IGNORECASE)
        m_st = re.search(r"step[=_]?(\d+)", fname, re.IGNORECASE)
        ep = int(m_ep.group(1)) if m_ep else 0
        st = int(m_st.group(1)) if m_st else 0
        return (st, ep)
    return max(ckpt_list, key=extract_epoch_step)

def find_local_ckpts():
    found = []
    for s_dir in (LOCAL_CHECKPOINTS, os.path.join(LOCAL_REPO, "lightning_logs")):
        if os.path.isdir(s_dir):
            found.extend(glob.glob(os.path.join(s_dir, "**", "*.ckpt"), recursive=True))
    valid = [
        f for f in sorted(found, key=os.path.getmtime)
        if not os.path.basename(f).endswith("_last.ckpt") and os.path.basename(f) not in ("last.ckpt", "resume_target.ckpt")
    ]
    return valid if valid else sorted(found, key=os.path.getmtime)

def get_latest_ckpt():
    local_ckpts = find_local_ckpts()
    if local_ckpts:
        target_file = pick_latest_checkpoint(local_ckpts)
    else:
        target_file = None

    if not target_file:
        hf_ckpt_files = [f for f in hf_list_files(HF_CHECKPOINTS_PREFIX) if f.endswith(".ckpt")]
        if not hf_ckpt_files:
            return None
        target_hf_file = pick_latest_checkpoint(hf_ckpt_files)
        print(f"Found remote checkpoint on Hugging Face: {target_hf_file}. Downloading...")
        target_file = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=target_hf_file,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )

    resume_path = os.path.join(LOCAL_CHECKPOINTS, "resume_target.ckpt")
    os.makedirs(LOCAL_CHECKPOINTS, exist_ok=True)
    if os.path.abspath(target_file) != os.path.abspath(resume_path):
        shutil.copy2(target_file, resume_path)
    return resume_path

_ARABIC_LETTERS_MAP = {
    "ء": "ʔ", "أ": "ʔ", "إ": "ʔ", "ؤ": "ʔ", "ئ": "ʔ", "آ": "ʔ aː",
    "ب": "b", "ت": "t", "ة": "t", "ث": "θ", "ج": "d͡ʒ", "ح": "ħ", "خ": "x",
    "د": "d", "ذ": "ð", "ر": "r", "ز": "z", "س": "s", "ش": "ʃ",
    "ص": "sˤ", "ض": "dˤ", "ط": "tˤ", "ظ": "ðˤ", "ع": "ʕ", "غ": "ɣ",
    "ف": "f", "ق": "q", "ك": "k", "ل": "l", "م": "m", "ن": "n",
    "ه": "h", "و": "w", "ي": "j", "ى": "aː", "ٱ": ""
}

_DIACRITICS_MAP = {
    "\\u064e": " a",
    "\\u064f": " u",
    "\\u0650": " i",
    "\\u064b": " a n",
    "\\u064c": " u n",
    "\\u064d": " i n",
    "\\u0652": "",
    "\\u0670": " aː"
}

_AR_DIGITS_MAP = {
    "0": "صِفْر", "1": "وَاحِد", "2": "اثْنَان", "3": "ثَلَاثَة", "4": "أَرْبَعَة",
    "5": "خَمْسَة", "6": "سِتَّة", "7": "سَبْعَة", "8": "ثَمَانِيَة", "9": "تِسْعَة",
    "٠": "صِفْر", "١": "وَاحِد", "٢": "اثْنَان", "٣": "ثَلَاثَة", "٤": "أَرْبَعَة",
    "٥": "خَمْسَة", "٦": "سِتَّة", "٧": "سَبْعَة", "٨": "ثَمَانِيَة", "٩": "تِسْعَة"
}

_EN_DIGITS_MAP = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
    "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"
}

def normalize_text_multilingual(text):
    text = unicodedata.normalize("NFC", text.strip())
    has_ar = bool(re.search(r"[\\u0600-\\u06ff]", text))
    if has_ar:
        for d, word in _AR_DIGITS_MAP.items():
            text = text.replace(d, f" {word} ")
    else:
        for d, word in _EN_DIGITS_MAP.items():
            text = text.replace(d, f" {word} ")
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def rule_based_arabic_phonemize(text):
    text = unicodedata.normalize("NFC", text.strip())
    text = re.sub(r"اللَّ?ه", "al lˤ aː h", text)
    tokens = []
    i = 0
    while i < len(text):
        c = text[i]
        if c == " ":
            tokens.append(" ")
            i += 1
            continue
        if c in _ARABIC_LETTERS_MAP:
            base_ph = _ARABIC_LETTERS_MAP[c]
            i += 1
            is_shadda = False
            diacritic_ph = ""
            while i < len(text) and text[i] in ("\\u0651", "\\u064e", "\\u064f", "\\u0650", "\\u064b", "\\u064c", "\\u064d", "\\u0652", "\\u0670"):
                d = text[i]
                if d == "\\u0651":
                    is_shadda = True
                elif d in _DIACRITICS_MAP:
                    diacritic_ph += _DIACRITICS_MAP[d]
                i += 1
            if base_ph:
                if is_shadda:
                    tokens.append(base_ph)
                tokens.append(base_ph)
            if diacritic_ph:
                tokens.extend(diacritic_ph.strip().split())
        elif c in _DIACRITICS_MAP:
            d_ph = _DIACRITICS_MAP[c]
            if d_ph:
                tokens.extend(d_ph.strip().split())
            i += 1
        elif c == "ا":
            if tokens and tokens[-1] == "a":
                tokens[-1] = "aː"
            else:
                tokens.append("aː")
            i += 1
        else:
            tokens.append(c)
            i += 1
    res = []
    for t in tokens:
        if t == "w" and res and res[-1] == "u":
            res[-1] = "uː"
        elif t == "j" and res and res[-1] == "i":
            res[-1] = "iː"
        elif t:
            res.append(t)
    return " ".join(res)

_IPA_SYMBOLS_RAW = [
    "_", "~", "<unk>", "<bos>", "<eos>", " ", "|", "sil", "sp",
    "!", "'", "(", "+", ")", ",", "-", ".", ":", ";", "?", "«", "»", "“", "”", "؛", "،", "؟", '"',
    "ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ",
    "p", "v", "g", "ɡ", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ",
    "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z",
    "a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ",
    "ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", ".", "̯", "̩", "͡"
]

_GLOBAL_SYMBOLS = []
_seen_syms = set()
for s in _IPA_SYMBOLS_RAW:
    if s not in _seen_syms:
        _GLOBAL_SYMBOLS.append(s)
        _seen_syms.add(s)

class MultilingualPhonemizerEngine:
    def __init__(self, symbols=_GLOBAL_SYMBOLS):
        self.symbols = symbols
        self.symbol_to_id = {s: i for i, s in enumerate(symbols)}
        self.id_to_symbol = {i: s for i, s in enumerate(symbols)}
        self.unk_id = self.symbol_to_id.get("<unk>", 2)
        self.backend_ar = None
        self.backend_en = None
        try:
            from phonemizer.backend import EspeakBackend
            self.backend_ar = EspeakBackend(language="ar", preserve_punctuation=True, with_stress=False)
        except Exception:
            pass
        try:
            from phonemizer.backend import EspeakBackend
            self.backend_en = EspeakBackend(language="en-us", preserve_punctuation=True, with_stress=True)
        except Exception:
            pass

    def phonemize_mixed(self, text):
        text = normalize_text_multilingual(text)
        tokens = text.split()
        result_parts = []
        for tok in tokens:
            has_ar = bool(re.search(r"[\\u0600-\\u06ff]", tok))
            backend = self.backend_ar if (has_ar and self.backend_ar) else self.backend_en
            phonemized_ok = False
            if backend:
                try:
                    ph = backend.phonemize([tok], strip=True)
                    if ph and ph[0]:
                        result_parts.append(ph[0])
                        phonemized_ok = True
                except Exception:
                    pass
            if not phonemized_ok:
                if has_ar:
                    result_parts.append(rule_based_arabic_phonemize(tok))
                else:
                    result_parts.append(tok)
        return " ".join(result_parts)

    def text_to_sequence(self, text):
        ipa = self.phonemize_mixed(text)
        ipa = unicodedata.normalize("NFC", ipa).lower()
        tokens = []
        i = 0
        while i < len(ipa):
            matched = False
            for length in (5, 4, 3, 2, 1):
                sub = ipa[i:i + length]
                if sub in self.symbol_to_id:
                    tokens.append(sub)
                    i += length
                    matched = True
                    break
            if not matched:
                tokens.append("<unk>")
                i += 1
        seq = [self.symbol_to_id.get(t, self.unk_id) for t in tokens]
        ipa_str = " ".join(tokens)
        return seq, ipa_str, tokens

phonemizer_engine = MultilingualPhonemizerEngine()
synthesis_lock = threading.Lock()

def synthesize_and_send_audio(chat_id, text_to_say):
    if not synthesis_lock.acquire(blocking=False):
        send_telegram_to_chat(chat_id, "جاري بالفعل معالجة طلب صوتي آخر، يرجى الانتظار بضع ثوانٍ...")
        return
    try:
        import yaml
        import soundfile as sf
        import numpy as np

        ckpts = find_local_ckpts()
        if not ckpts:
            send_telegram_to_chat(chat_id, "لا يوجد نقطة فحص (checkpoint) جاهزة بعد. يرجى الانتظار لحفظ أول نقطة فحص.")
            return

        latest_c = ckpts[-1]
        ep_name = os.path.basename(latest_c)
        send_telegram_to_chat(chat_id, f"جاري توليد الصوت باستخدام {ep_name}...\nالنص: {text_to_say}")

        from model_l1_ssim_gvar import GrainSpeech

        with open(config_yaml_path, "r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)

        stats_dir = LOCAL_DATA_STATS
        if not os.path.exists(os.path.join(stats_dir, "stats.json")):
            alt_dir = os.path.join(LOCAL_REPO, "configs", "Kawthar")
            if os.path.exists(os.path.join(alt_dir, "stats.json")):
                stats_dir = alt_dir
        cfg["path"]["preprocessed_path"] = stats_dir

        infer_dev = "cpu"
        model = GrainSpeech(preprocess_config=cfg, hifigan_checkpoint=HIFIGAN_CKPT, infer_device=infer_dev)
        ckpt_data = torch.load(latest_c, map_location=infer_dev, weights_only=False)
        model.load_state_dict(ckpt_data.get("state_dict", ckpt_data), strict=False)
        model.eval().to(infer_dev)

        seq, ipa_str, _ = phonemizer_engine.text_to_sequence(text_to_say)
        if not seq:
            send_telegram_to_chat(chat_id, "تعذر تحويل النص إلى تسلسل فونيمات صالح.")
            return

        in_tensor = torch.tensor([seq], dtype=torch.long, device=infer_dev)
        batch = {
            "phoneme": in_tensor,
            "phoneme_mask": torch.zeros_like(in_tensor, dtype=torch.bool)
        }

        with torch.no_grad():
            wav, mel_len, mel = model.predict_step(batch)

        hop_len = cfg["preprocessing"]["stft"]["hop_length"]
        sr_rate = cfg["preprocessing"]["audio"]["sampling_rate"]
        sample_count = int(mel_len[0].item()) * hop_len
        audio_wav = wav[0, :sample_count].float().cpu().numpy()
        audio_wav = np.clip(audio_wav * 32768.0, -32768, 32767).astype(np.int16)

        out_wav = os.path.join(KAGGLE_WORKING, f"telegram_speech_{int(time.time())}.wav")
        sf.write(out_wav, audio_wav, sr_rate)

        caption = f"GrainSpeech TTS ({ep_name})\n{text_to_say[:200]}"
        url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendAudio"
        with open(out_wav, "rb") as af:
            requests.post(url, data={"chat_id": chat_id, "caption": caption}, files={"audio": af}, timeout=30)

        try:
            os.remove(out_wav)
        except Exception:
            pass
    except Exception as e:
        send_telegram_to_chat(chat_id, f"خطأ أثناء توليد الصوت: {e}")
    finally:
        synthesis_lock.release()

if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
grainspeech_pkg = os.path.join(LOCAL_REPO, "grainspeech")
if grainspeech_pkg not in sys.path:
    sys.path.insert(0, grainspeech_pkg)
os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

subprocess.run(["git", "checkout", "--", "."], cwd=LOCAL_REPO, check=False)

repo_symbols_patch = '''_pad = "_"
_blank = "~"
_unk = "<unk>"
_bos = "<bos>"
_eos = "<eos>"
_space = " "
_word_boundary = "|"
_silence = ["sil", "sp"]
_punctuation = list("!'(+),-.:;? «»“”؛،؟") + ['"']
_arabic_ipa = ["ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ"]
_english_ipa = ["p", "v", "g", "ɡ", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ"]
_latin_letters = list("abcdefghijklmnopqrstuvwxyz")
_vowels = ["a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ"]
_modifiers = ["ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", ".", "̯", "̩", "͡"]
symbols = []
seen = set()
for s in ([_pad, _blank, _unk, _bos, _eos, _space, _word_boundary] + _silence + _punctuation + _arabic_ipa + _english_ipa + _latin_letters + _vowels + _modifiers):
    if s not in seen:
        symbols.append(s)
        seen.add(s)
'''
for sym_name in ("symbols.py", "symbols_exp.py"):
    for base_p in (LOCAL_REPO, os.path.join(LOCAL_REPO, "grainspeech")):
        target_sym = os.path.join(base_p, "text", sym_name)
        if os.path.exists(os.path.dirname(target_sym)):
            with open(target_sym, "w", encoding="utf-8") as f:
                f.write(repo_symbols_patch.strip() + "\n")

for ti_f in (os.path.join(LOCAL_REPO, "grainspeech", "text", "__init__.py"), os.path.join(LOCAL_REPO, "text", "__init__.py")):
    if os.path.exists(ti_f):
        with open(ti_f, "r", encoding="utf-8") as f:
            ti_txt = f.read()
        if "unk_id = _symbol_to_id.get" not in ti_txt:
            patch_str = '''
def text_to_sequence(text, cleaner_names):
    import unicodedata
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        raw_tokens = text[1:-1].split()
    elif "{" in text and "}" in text:
        m = re.search(r"[{](.+?)[}]", text)
        raw_tokens = m.group(1).split() if m else text.split()
    else:
        raw_tokens = text.split()
    unk_id = _symbol_to_id.get("<unk>", 2)
    seq = []
    for t in raw_tokens:
        clean_t = unicodedata.normalize("NFC", t.strip())
        if not clean_t:
            continue
        if clean_t in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t])
        elif clean_t.startswith("@") and clean_t[1:] in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t[1:]])
        else:
            for ch in clean_t:
                seq.append(_symbol_to_id.get(ch, unk_id))
    return seq
'''
            with open(ti_f, "a", encoding="utf-8") as f:
                f.write(patch_str)

pad_target = '''        s = np.shape(x)[1]
        x_padded = np.pad(
            x, (0, max_len - np.shape(x)[0]), mode="constant", constant_values=PAD
        )
        return x_padded[:, :s]'''
pad_patch = '''        return np.pad(
            x, ((0, max_len - np.shape(x)[0]), (0, 0)), mode="constant", constant_values=PAD
        )'''

for tp_f in (os.path.join(LOCAL_REPO, "grainspeech", "utils", "tools.py"), os.path.join(LOCAL_REPO, "utils", "tools.py")):
    if os.path.exists(tp_f):
        with open(tp_f, "r", encoding="utf-8") as f:
            tp_txt = f.read()
        if pad_target in tp_txt:
            tp_txt = tp_txt.replace(pad_target, pad_patch)
        with open(tp_f, "w", encoding="utf-8") as f:
            f.write(tp_txt)

for mp_name in ("model_l1_ssim_gvar.py", "model_l1_ssim.py", "model.py"):
    for base_p in (LOCAL_REPO, os.path.join(LOCAL_REPO, "grainspeech")):
        mp = os.path.join(base_p, mp_name)
        if os.path.exists(mp):
            with open(mp, "r", encoding="utf-8") as f:
                m_code = f.read()
            if "def on_train_epoch_start" not in m_code and "    def on_train_epoch_end" in m_code:
                m_code = m_code.replace("    def on_train_epoch_end", "    def on_train_epoch_start(self):\n        self.training_step_outputs = []\n\n    def on_train_epoch_end")
            old_target = "    def on_train_epoch_end(self):\n        avg_loss = torch.stack([x[\"loss\"] for x in self.training_step_outputs]).mean()"
            new_target = "    def on_train_epoch_end(self):\n        if not hasattr(self, \"training_step_outputs\") or not self.training_step_outputs:\n            return\n        avg_loss = torch.stack([x[\"loss\"] for x in self.training_step_outputs]).mean()"
            if old_target in m_code:
                m_code = m_code.replace(old_target, new_target)
            leak_target = "self.training_step_outputs.append(losses)"
            clean_target = "self.training_step_outputs.append({k: v.detach() if hasattr(v, 'detach') else v for k, v in losses.items()})"
            if leak_target in m_code:
                m_code = m_code.replace(leak_target, clean_target)
            old_lr = 'self.log("lr", self.scheduler.get_last_lr()[0], on_epoch=True, prog_bar=True, sync_dist=True)'
            new_lr = 'self.log("lr", self.scheduler.get_last_lr()[0], on_epoch=True, prog_bar=True, sync_dist=False)'
            if old_lr in m_code:
                m_code = m_code.replace(old_lr, new_lr)
            with open(mp, "w", encoding="utf-8") as f:
                f.write(m_code)

train_script_p = os.path.join(LOCAL_REPO, "grainspeech", "train_l1_ssim_gvar.py")
if os.path.exists(train_script_p):
    with open(train_script_p, "r", encoding="utf-8") as f:
        ts_code = f.read()
    if "weights_only" not in ts_code:
        ts_code = "import torch\nif hasattr(torch, 'load'):\n    _orig_l = torch.load\n    def _compat_l(*a, **k):\n        k['weights_only'] = False\n        return _orig_l(*a, **k)\n    torch.load = _compat_l\n" + ts_code
    if "CleanProgressCallback" not in ts_code:
        cb_code = '''
import gc
import ctypes
from lightning.pytorch.callbacks import Callback

class CleanProgressCallback(Callback):
    def __init__(self):
        super().__init__()
        try:
            self._libc = ctypes.CDLL("libc.so.6")
        except Exception:
            self._libc = None

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        loss_val = 0.0
        if outputs is not None:
            if isinstance(outputs, torch.Tensor):
                loss_val = float(outputs.detach().cpu().item())
            elif isinstance(outputs, dict) and "loss" in outputs:
                v = outputs["loss"]
                loss_val = float(v.detach().cpu().item() if hasattr(v, "item") else v)
        if loss_val == 0.0 and hasattr(pl_module, "training_step_outputs"):
            outs = getattr(pl_module, "training_step_outputs", None)
            if outs and len(outs) > 0:
                v = outs[-1].get("loss") if isinstance(outs[-1], dict) else outs[-1]
                if v is not None:
                    loss_val = float(v.detach().cpu().item() if hasattr(v, "item") else v)
        if loss_val == 0.0:
            _m = trainer.callback_metrics.get("loss")
            if _m is not None:
                loss_val = float(_m)
        del outputs, batch
        total = getattr(trainer, "num_training_batches", 0)
        cur = batch_idx + 1
        if cur % 25 == 0 or cur == total:
            gc.collect()
            if self._libc and hasattr(self._libc, "malloc_trim"):
                try:
                    self._libc.malloc_trim(0)
                except Exception:
                    pass
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if cur % 5 == 0 or cur == total:
            pct = (cur / total) * 100.0 if total else 0.0
            print(f"PROGRESS: Epoch {trainer.current_epoch + 1}: {pct:.1f}% | Step {cur}/{total} | Loss: {loss_val:.4f}", flush=True)

    def on_train_epoch_end(self, trainer, pl_module):
        gc.collect()
        if self._libc and hasattr(self._libc, "malloc_trim"):
            try:
                self._libc.malloc_trim(0)
            except Exception:
                pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        avg_loss = trainer.callback_metrics.get("loss")
        val_loss = f"{avg_loss.item():.4f}" if avg_loss is not None else ""
        print(f"EPOCH_END: Epoch {trainer.current_epoch + 1} completed | Loss: {val_loss}", flush=True)
'''
        ts_code = cb_code + "\n" + ts_code
        target_trainer = "callbacks=[checkpoint_callback],"
        repl_trainer = "enable_progress_bar=False, callbacks=[checkpoint_callback, CleanProgressCallback()],"
        ts_code = ts_code.replace(target_trainer, repl_trainer)
    with open(train_script_p, "w", encoding="utf-8") as f:
        f.write(ts_code)

latest_ckpt = get_latest_ckpt()

try:
    subprocess.run("pkill -9 -f 'train_l1_ssim_gvar' ; pkill -9 -f 'grainspeech' ; pkill -9 -f 'torch.distributed'", shell=True, check=False)
    time.sleep(1)
except Exception:
    pass

for tmp_pat in ("/tmp/*.tar*", "/tmp/*.zip", "/tmp/*.npy", "/tmp/preprocessed_data*", "/tmp/wavs*"):
    for tmp_f in glob.glob(tmp_pat):
        try:
            os.remove(tmp_f)
        except Exception:
            pass

import gc
gc.collect()

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "100000"

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        torch.cuda.ipc_collect()
    device_desc = f"{torch.cuda.get_device_name(0)} (Single GPU)"
    accelerator_choice = "gpu"
    devices_count = 1
    workers_per_gpu = 0
    actual_batch_size = min(32, BATCH_SIZE)
else:
    device_desc = "CPU"
    accelerator_choice = "cpu"
    devices_count = 1
    workers_per_gpu = 0
    actual_batch_size = min(16, BATCH_SIZE)

train_cmd = [
    sys.executable, "-u", "grainspeech/train_l1_ssim_gvar.py",
    "--run-name", EXPERIMENT_NAME,
    "--preprocess-config", config_yaml_path,
    "--hifigan-checkpoint", HIFIGAN_CKPT,
    "--accelerator", accelerator_choice,
    "--devices", str(devices_count),
    "--precision", OPTIMAL_PRECISION,
    "--batch-size", str(actual_batch_size),
    "--lr", "0.001",
    "--weight-decay", "0.00001",
    "--num_workers", str(workers_per_gpu),
    "--max_epochs", "5000",
    "--infer-device", "cuda" if torch.cuda.is_available() else "cpu",
]

if latest_ckpt:
    print(f"[GrainSpeech] Resuming training from checkpoint: {latest_ckpt}")
    train_cmd.extend(["--checkpoint", latest_ckpt])
else:
    print("[GrainSpeech] Starting training from scratch (no previous checkpoint found).")

current_progress = {
    "step": 0,
    "epoch": 0,
    "loss": "",
    "pct": "0.0%"
}
last_progress_line = [""]
recent_lines = []

def format_single_last_ckpt_name(ckpt_path):
    fname = os.path.basename(ckpt_path)
    m_ep = re.search(r"epoch[=_]?(\d+)", fname, re.IGNORECASE)
    m_st = re.search(r"step[=_]?(\d+)", fname, re.IGNORECASE)
    if m_ep and m_st:
        ep = int(m_ep.group(1))
        st = int(m_st.group(1))
        return f"epoch_{ep}_step_{st}_last.ckpt"
    try:
        data = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        ep = data.get("epoch", 0)
        st = data.get("global_step", 0)
        if ep or st:
            return f"epoch_{ep}_step_{st}_last.ckpt"
    except Exception:
        pass
    return "epoch_last.ckpt"

def sync_checkpoint_to_hf(local_path):
    if not ACTIVE_HF_TOKEN or not hf_api:
        return
    ckpt_name = format_single_last_ckpt_name(local_path)
    remote_path = f"{HF_CHECKPOINTS_PREFIX}/{ckpt_name}"
    success = hf_upload_file(local_path, remote_path)
    if success:
        broadcast_telegram(f"تم حفظ ورفع Checkpoint بنجاح:\nالملف: {ckpt_name}\nالإيبوك: {current_progress['epoch']} | الخسارة: {current_progress['loss']}")
        try:
            hf_files = [f for f in hf_list_files(HF_CHECKPOINTS_PREFIX) if f.endswith(".ckpt")]
            if len(hf_files) > 2:
                dated = sorted(hf_files, key=lambda x: os.path.basename(x))
                to_del = [f for f in dated[:-2] if not f.endswith("last.ckpt")]
                if to_del:
                    hf_delete_files(to_del)
        except Exception:
            pass

def poll_telegram_commands(stop_event):
    while not stop_event.is_set():
        try:
            updates = get_telegram_updates()
            for u in updates:
                msg = u.get("message", {})
                chat_id = msg.get("chat", {}).get("id")
                raw_text = (msg.get("text") or "").strip()
                if not chat_id or not raw_text:
                    continue
                cmd = raw_text.split()[0].lower() if raw_text else ""
                if cmd in ("/start", "/help", "help", "مساعدة", "اوامر"):
                    help_msg = (
                        "مساعد GrainSpeech الذكي للمراقبة وتوليد الصوت\n\n"
                        "الأوامر المتاحة:\n"
                        "/sample - توليد عينة عربية مشكولة حية بصوت النموذج الآن\n"
                        "/say <نص> - تحويل أي نص عربي مشكول أو إنجليزي إلى كلام فوري\n"
                        "/status - عرض تقدم التدريب الحالي (الإيبوك، الخطوة، Loss)\n"
                        "/logs - عرض آخر سجلات التدريب\n"
                        "/save - رفع أحدث Checkpoint إلى Hugging Face فوراً\n"
                        "/checkpoints - اسم أحدث نقطة فحص محفوظة\n"
                        "/stop - إيقاف التدريب وحفظ الحالة الحالية\n\n"
                        "ملاحظة: يمكنك إرسال أي نص مشكول مباشرة إلى المحادثة وسيقوم البوت بنطقه فوراً!"
                    )
                    send_telegram_to_chat(chat_id, help_msg)
                elif cmd == "/status":
                    vram_txt = ""
                    if torch.cuda.is_available():
                        alloc = torch.cuda.memory_allocated(0) / (1024**3)
                        res = torch.cuda.memory_reserved(0) / (1024**3)
                        vram_txt = f"\nاستهلاك الـ VRAM: {alloc:.2f} GB / {res:.2f} GB"
                    status_msg = (
                        f"حالة التدريب:\n"
                        f"الإيبوك: {current_progress['epoch']}\n"
                        f"الخطوة: {current_progress['step']}\n"
                        f"التقدم: {current_progress['pct']}\n"
                        f"الخسارة: {current_progress['loss']}"
                        f"{vram_txt}"
                    )
                    send_telegram_to_chat(chat_id, status_msg)
                elif cmd in ("/sample", "/voice", "/test", "عينة", "صوت"):
                    sample_text = "الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ، الرَّحْمٰنِ الرَّحِيمِ، مَالِكِ يَوْمِ الدِّينِ"
                    threading.Thread(target=synthesize_and_send_audio, args=(chat_id, sample_text), daemon=True).start()
                elif cmd.startswith("/say") or cmd.startswith("/tts") or cmd.startswith("/speak"):
                    parts = raw_text.split(maxsplit=1)
                    if len(parts) > 1 and parts[1].strip():
                        threading.Thread(target=synthesize_and_send_audio, args=(chat_id, parts[1].strip()), daemon=True).start()
                    else:
                        sample_text = "الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ، الرَّحْمٰنِ الرَّحِيمِ"
                        send_telegram_to_chat(chat_id, f"لم تحدد نصاً، جاري توليد عينة عربية مشكولة افتراضية:\n{sample_text}\n\nيمكنك كتابة أي نص مخصص بعد الأمر، مثل:\n/say مَرْحَبًا بِكُمْ فِي نَمُوذَجِ جِرِين سْبِيتْش")
                        threading.Thread(target=synthesize_and_send_audio, args=(chat_id, sample_text), daemon=True).start()
                elif cmd == "/loss":
                    send_telegram_to_chat(chat_id, f"آخر خسارة: {current_progress['loss']}")
                elif cmd == "/logs":
                    tail = "\n".join(recent_lines[-25:])
                    send_telegram_to_chat(chat_id, f"آخر السجلات:\n{tail[:3500]}")
                elif cmd == "/save":
                    send_telegram_to_chat(chat_id, "جاري رفع أحدث Checkpoint فوراً إلى Hugging Face...")
                    local_ckpts = find_local_ckpts()
                    if local_ckpts:
                        sync_checkpoint_to_hf(local_ckpts[-1])
                    else:
                        send_telegram_to_chat(chat_id, "لا توجد نقاط فحص محلية بعد.")
                elif cmd == "/checkpoints":
                    local_ckpts = find_local_ckpts()
                    c_name = os.path.basename(local_ckpts[-1]) if local_ckpts else "لا يوجد"
                    send_telegram_to_chat(chat_id, f"أحدث Checkpoint:\n{c_name}")
                elif cmd == "/stop":
                    send_telegram_to_chat(chat_id, "جاري إيقاف التدريب وحفظ الحالة...")
                    subprocess.run("pkill -9 -f 'train_l1_ssim_gvar'", shell=True, check=False)
                elif not raw_text.startswith("/") and len(raw_text.strip()) > 1:
                    threading.Thread(target=synthesize_and_send_audio, args=(chat_id, raw_text.strip()), daemon=True).start()
        except Exception:
            pass
        time.sleep(2)

def periodic_hf_sync(stop_event):
    last_synced_mtime = 0
    last_logs_sync = 0
    while not stop_event.is_set():
        try:
            local_ckpts = find_local_ckpts()
            if local_ckpts:
                newest = local_ckpts[-1]
                mtime = os.path.getmtime(newest)
                if mtime > last_synced_mtime:
                    sync_checkpoint_to_hf(newest)
                    last_synced_mtime = mtime
            now = time.time()
            if os.path.isdir(LOCAL_LOGS) and (now - last_logs_sync >= 600):
                log_files = glob.glob(os.path.join(LOCAL_LOGS, "**/*"), recursive=True)
                latest_log_mtime = max([os.path.getmtime(f) for f in log_files if os.path.isfile(f)], default=0)
                if latest_log_mtime > last_logs_sync:
                    hf_upload_folder(LOCAL_LOGS, HF_LOGS_PREFIX)
                last_logs_sync = now
        except Exception:
            pass
        for _ in range(30):
            if stop_event.is_set():
                break
            time.sleep(1)

tg_stop_event = threading.Event()
tg_thread = threading.Thread(target=poll_telegram_commands, args=(tg_stop_event,), daemon=True)
tg_thread.start()

sync_stop_event = threading.Event()
sync_thread = threading.Thread(target=periodic_hf_sync, args=(sync_stop_event,), daemon=True)
sync_thread.start()

broadcast_telegram(f" تم بدء جلسة التدريب GrainSpeech بنجاح على {device_desc}!\nBatch Size: {actual_batch_size}\nPrecision: {OPTIMAL_PRECISION}")

print(f"Executing: {' '.join(train_cmd)}")

proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=LOCAL_REPO
)

last_print_len = 0
for line in proc.stdout:
    line_str = line.strip()
    if not line_str:
        continue
    recent_lines.append(line_str)
    if len(recent_lines) > 20:
        recent_lines.pop(0)

    if line_str.startswith("PROGRESS:"):
        m = re.search(r"Epoch\s+(\d+):\s*([\d\.]+)%\s*\|\s*Step\s+(\d+)/(\d+)\s*\|\s*Loss:\s*([\d\.]+)", line_str)
        if m:
            current_progress["epoch"] = int(m.group(1))
            current_progress["pct"] = f"{m.group(2)}%"
            current_progress["step"] = f"{m.group(3)}/{m.group(4)}"
            current_progress["loss"] = m.group(5)
        disp = f"\r[TRAIN] Epoch {current_progress['epoch']} | Step {current_progress['step']} ({current_progress['pct']}) | Loss: {current_progress['loss']} "
        pad = " " * max(0, last_print_len - len(disp))
        sys.stdout.write(disp + pad)
        sys.stdout.flush()
        last_print_len = len(disp)
    elif line_str.startswith("EPOCH_END:"):
        m = re.search(r"Epoch\s+(\d+)\s+completed\s*\|\s*Loss:\s*([\d\.]*)", line_str)
        if m:
            current_progress["epoch"] = int(m.group(1))
            if m.group(2):
                current_progress["loss"] = m.group(2)
        disp = f"\r[TRAIN] Epoch {current_progress['epoch']} Complete | Loss: {current_progress['loss']} | Starting Next Epoch... "
        pad = " " * max(0, last_print_len - len(disp))
        sys.stdout.write(disp + pad)
        sys.stdout.flush()
        last_print_len = len(disp)
    else:
        if any(ign in line_str for ign in ("v_num=", "GPU available:", "TPU available:", "Using 16bit", "Restoring states", "LOCAL_RANK:")):
            continue
        if any(err_kw in line_str for err_kw in ("Traceback", "Error", "Exception", "CUDA out of memory")):
            print(f"\n[ALERT] {line_str}", flush=True)
            last_print_len = 0
        elif "Resuming full training state" in line_str or "Training from random" in line_str or "network file:" in line_str:
            print(f"\n{line_str}", flush=True)
            last_print_len = 0

ret = proc.wait()

tg_stop_event.set()
sync_stop_event.set()

local_ckpts = find_local_ckpts()
if local_ckpts:
    sync_checkpoint_to_hf(local_ckpts[-1])
if os.path.isdir(LOCAL_LOGS):
    hf_upload_folder(LOCAL_LOGS, HF_LOGS_PREFIX)

if ret == 0:
    print("\nTraining completed successfully!")
    broadcast_telegram(f" اكتمل تدريب GrainSpeech بنجاح!\nالإيبوك: {current_progress['epoch']} | الخسارة النهائية: {current_progress['loss']}")
else:
    print(f"\n[ERROR] Process terminated with exit code {ret}.")
    recent_tail = "\n".join(recent_lines[-10:])
    broadcast_telegram(f" توقف التدريب بسبب خطأ (Code {ret})!\nآخر المخرجات:\n{recent_tail[:300]}")

In [ ]:
import shutil
import subprocess
import glob
import sys
import os
QUANTIZE_INT8 = False
TEST_TEXT = "مرحبا بكم في تجربة نموذج جرين سبيتش لتحويل النص الى كلام عالي الجودة"

all_c = find_local_ckpts() if "find_local_ckpts" in dir() else glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
checkpoint_path = all_c[-1] if all_c else None

if checkpoint_path and os.path.exists(checkpoint_path):
    print(f"Loading checkpoint for inference & export: {checkpoint_path}")
    from model_l1_ssim_gvar import GrainSpeech, get_hifigan
    from text import text_to_sequence
    import yaml
    from IPython.display import Audio, display

    with open(config_yaml_path, "r", encoding="utf-8") as f:
        preprocess_config = yaml.safe_load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        model = GrainSpeech.load_from_checkpoint(
            checkpoint_path,
            preprocess_config=preprocess_config,
            hifigan_checkpoint=HIFIGAN_CKPT,
            infer_device=device,
            map_location=device,
            weights_only=False,
        )
    except Exception:
        model = GrainSpeech(
            preprocess_config=preprocess_config,
            hifigan_checkpoint=HIFIGAN_CKPT,
            infer_device=device,
        )
        ckpt_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
        st = ckpt_data.get("state_dict", ckpt_data)
        model.load_state_dict(st, strict=False)

    model.eval().to(device)
    vocoder = get_hifigan(checkpoint=HIFIGAN_CKPT, infer_device=device)

    cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
    seq = text_to_sequence(TEST_TEXT, cleaners)
    if seq:
        in_tensor = torch.tensor([seq], dtype=torch.long, device=device)
        with torch.no_grad():
            pred = model.phoneme2mel(in_tensor)
            mel = pred[1] if isinstance(pred, (list, tuple)) else (pred["mel"] if isinstance(pred, dict) else pred)
            if mel.dim() == 3:
                mel = mel.transpose(1, 2)
            if vocoder is not None:
                wav = vocoder(mel).squeeze().cpu().numpy()
                display(Audio(wav, rate=SAMPLE_RATE))

    class GrainSpeechOnnxExport(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m.phoneme2mel

        def forward(self, phonemes):
            out = self.m(phonemes)
            mel = out[1] if isinstance(out, (list, tuple)) else (out["mel"] if isinstance(out, dict) else out)
            return mel

    onnx_file = os.path.join(LOCAL_ONNX_EXPORT, "grainspeech_kawthar.onnx")
    wrapper = GrainSpeechOnnxExport(model)
    dummy_input = torch.randint(1, 80, (1, 30), dtype=torch.long, device=device)

    try:
        torch.onnx.export(
            wrapper,
            (dummy_input,),
            onnx_file,
            input_names=["phonemes"],
            output_names=["mel"],
            dynamic_axes={"phonemes": {1: "seq_len"}, "mel": {1: "time_frames"}},
            opset_version=14,
        )
        old_onnx = hf_list_files(HF_ONNX_PREFIX)
        if old_onnx:
            hf_delete_files(old_onnx)
        hf_upload_folder(LOCAL_ONNX_EXPORT, HF_ONNX_PREFIX)
        print("Cell 8 Complete: ONNX models exported & uploaded to Hugging Face successfully.")
    except Exception as e:
        print(f"ONNX export notice: {e}")
else:
    print("No checkpoint found for ONNX export.")
